# 各固有名詞について、その特徴をwikiページから抽出する。

In [ ]:
import pandas as pd
import json
import re
import os
import sys
from tqdm import tqdm
import wikipediaapi
from dotenv import load_dotenv


# プロジェクトのutils追加
project_root = os.path.join(os.getcwd(), "..")
sys.path.append(project_root)


load_dotenv() # API keyの読み込み

# from openai import OpenAI
# client = OpenAI()

# gptは少し高いのでgeminiに変更。wikiからの特徴抽出タスクは複雑な思考を必要としないため、軽量のモデルで十分と判断。
from google import genai
client = genai.Client()

## 固有名詞読み込み

In [ ]:

delete_list = ["year", "Election"] # year は、すべて2001, 2020, 2231 のような年の数字のみだったため削除。Electionも実体ではない

whole_df = pd.DataFrame()
for filename in os.listdir(os.path.join(project_root, "data", "dbpedia", "wikidata_Things_childs_LIMIT1000")):
    if filename.replace(".csv", "") in delete_list:
        # 削除対象のカテゴリは読み込まない
        continue
    if filename.endswith(".csv"):
        df = pd.read_csv(os.path.join(project_root, "data", "dbpedia", "wikidata_Things_childs_LIMIT1000", filename))
        whole_df = pd.concat([whole_df, df], ignore_index=True)
print(f"Total proper nouns collected: {len(whole_df)}")
whole_df.head()

Total proper nouns collected: 158013


,qid,label,class_label,class_qid
0,Q137919338,Colombian National Cross-Country Skiing,swimmer,Q10843402
1,Q12512,St. Peter's Basilica,church,Q16970
2,Q48435,Basilica and Expiatory Church of the Holy Family,church,Q16970
3,Q5933,Westminster Abbey,church,Q16970
4,Q2943,Sistine Chapel,church,Q16970


### 複数行に同じlabelがある場合は、そのlabelの行を削除する
- subclassのsubclassを取っているものもあるため、上位のsubclassとその下位のsubclassの両方に同じ固有名詞が属している心配がある。


In [3]:
# 複数行に同じlabelがある場合は、そのlabelの行を削除する
# whole_filtered_df 
whole_df = whole_df.drop_duplicates(subset=['label'], keep=False)
# print(f"Total proper nouns after removing duplicates: {len(whole_filtered_df)}")
print(f"Total proper nouns after removing duplicates: {len(whole_df)}")

Total proper nouns after removing duplicates: 123174


### 属する固有名詞が一定数以上ある中カテゴリを、実験対象として選ぶ

In [4]:
# 中カテゴリごとの固有名詞の数をカウントし、dfにする
category_count_df = whole_df.groupby("class_label").size().reset_index(name="count")
category_count_df

,class_label,count
0,Algorithm,419
1,Archive,366
2,Band,939
3,Biomolecule,7
4,Chief,5
...,...,...
234,weapon,262
235,wine,168
236,winery,623
237,written work,909


In [5]:
# propnoun_num_threshold 以上の固有名詞があるカテゴリを抽出する
# propnoun_num_threshold = 200 # 150
propnoun_num_for_init_vec = 100 # 1カテゴリあたりで、初期化vec(平均vec)の作成に使う概念数
propnoun_num_for_new_concept = 50 # 1カテゴリあたりで、新規概念の元にする概念数
propnoun_num_threshold = propnoun_num_for_init_vec + propnoun_num_for_new_concept
filtered_category_count_df = category_count_df[category_count_df["count"] >= propnoun_num_threshold]
print(f"{propnoun_num_threshold}以上の固有名詞があるカテゴリの数: {filtered_category_count_df.shape[0]}")
filtered_category_count_df

150以上の固有名詞があるカテゴリの数: 159


,class_label,count
0,Algorithm,419
1,Archive,366
2,Band,939
9,Food,985
11,Intercommunality,558
...,...,...
234,weapon,262
235,wine,168
236,winery,623
237,written work,909


### 大体のコストを試算

In [6]:
prompt_text_path = os.path.join(project_root, "data", "prompt", "baseprompt_of_gen_wiki_fact_extraction.txt")
with open(prompt_text_path, "r", encoding="utf-8") as f:
    gen_triplet_prompt_base = f.read()
print(len(gen_triplet_prompt_base.split())*1.5+400, "tokens in the base prompt for generating factual feature sentences") # 単語がsubtokenに分かれることで大体1.5倍くらいに増えると考えて想定

1286.5 tokens in the base prompt for generating factual feature sentences


In [ ]:

# gpt-5-miniの場合:
# cost = (0.25 / 1000000 * 1228.0 + 2 / 1000000 *4000) * 50 * 160
# gemini-2.5-flash-lightの場合:
cost = (0.1 / 1000000 * 1228.0 + 0.4 / 1000000 * 4000) * 50 * 160
print(f"Estimated cost for generating factual feature sentences for {propnoun_num_for_init_vec} categories, and each category has {propnoun_num_for_new_concept} concepts in one category: ${cost:.4f}")

Estimated cost for generating factual feature sentences for 100 categories, and each category has 50 concepts in one category: $13.7824


In [38]:
target_categories = filtered_category_count_df["class_label"].tolist()
print("=== Target categories for generating factual feature sentences ===")
print(target_categories)

=== Target categories for generating factual feature sentences ===
['Algorithm', 'Archive', 'Band', 'Food', 'Intercommunality', 'Legal Case', 'Medicine', 'Noble family', 'Painting', 'Port', 'Site of Special Scientific Interest', 'Wikimedia template', 'Windmill', 'academic subject', 'activity', 'agent', 'aircraft', 'airline', 'airport', 'album', 'anatomical structure', 'animal', 'archipelago', 'artery', 'artwork', 'asteroid', 'award', 'board game', 'book', 'bridge', 'broadcast network', 'broadcaster', 'brown dwarf', 'building', 'card game', 'casino', 'castle', 'cave', 'cemetery', 'church', 'city', 'company', 'contest', 'cycling race', 'dam', 'deity', 'drama', 'drug', 'educational institution', 'event', 'factory', 'family', 'fictional character', 'film festival', 'galaxy', 'game', 'garden', 'given name', 'glacier', 'golf course', 'government agency', 'handball team', 'historical period', 'hockey team', 'hot spring', 'hotel', 'identifier', 'ideology', 'island', 'lake', 'language', 'law fi

In [58]:
i = 10
print(target_categories[i])
concepts = whole_df[whole_df["class_label"] == target_categories[i]]["label"].tolist()
print(f"=== Example concepts in the category '{target_categories[i]}' ===")
print(concepts)

Site of Special Scientific Interest
=== Example concepts in the category 'Site of Special Scientific Interest' ===
['River Lugg', 'Erwood', 'Wye Valley', 'Glamorganshire Canal', 'Portsmouth Harbour', 'Severn Estuary', 'Breydon Water', 'Cemlyn Bay and lagoon', 'Ogof Ffynnon Ddu', 'Cosmeston Lakes Country Park', 'Graig Wood', 'Trewent Point', 'Newbourne Springs', 'Ditchling Common', 'Angle Peninsula Coast', 'Lochs of Spiggie and Brow', 'Castell Coch Woodlands and Road Section', 'Hartlepool Submerged Forest', 'Ynys Feurig', 'Malltraeth Marsh', "Devil's Punch Bowl", 'Crymlyn Bog', 'Cwmsymlog', 'Dinefwr Park National Nature Reserve', "Freeman's Marsh", 'Nant Llech', 'Penarth Coast', 'Penarth Quarry', 'Pentwyn Farm Grasslands', 'Southerham Grey Pit', 'Totternhoe Knolls', 'Arfordir Gogleddol Penmon', 'Beacon Hill', 'Bryn Euryn', 'Potteric Carr', 'Newborough Warren', 'Newport Wetlands', 'Leck Beck Head Catchment Area', 'Dwrhyd Pit', 'Gallt Llanerch - Coed Gelli-deg', 'Gruinart Flats SSSI', "Ha

In [42]:
t = """
In 1958, municipal associations on both sides of the border organised the first cross-border conference. An association on the German side, the ‘Interessensgemeinschaft Rhein-Ems’, had been founded as an intermunicipal interest association in 1954 by representatives of local business and local authority politicians of the Westmünsterland and the Grafschaft Bentheim and Lingen Kreise. Among its objectives were the improvement of the local and regional infrastructures, which, in the eyes of the local elites, deserved more attention on the part of the Land and federal governments. On the Dutch side of the border, this was followed by the establishment of its own inter-municipal association.

In 1962, the more informal Interessensgemeinschaft was replaced by the ‘Kommunalgemeinschaft Weser-Ems’, a public law intermunicipal body. A similar process of intermunicipal integration occurred in the adjoining Dutch border area where the municipalities in the areas of Twente and Gelderland founded the Belangengemeenschap Twente-Gelderland (later Samenwerkingsverband Twente, today Regio Twente) with the explicit objective of co-operating more closely with the Interessensgemeinschaft Rhein-Ems (later Kommunalgemeinschaft Rhein-Ems). Subsequently, a second association, the Samenwerkingsverband Oost-Gelderland (today Regio Achterhoek), was established in the Dutch border area. Today, the Kommunalgemeinschaft Rhein-Ems, the Regio Twente and the Regio Achterhoek together form the EUREGIO.

The EUREGIO Arbeitsgruppe (‘work group’) was founded in 1966 to operate as the informal board of the cross-border region. On the basis of regular meetings, it attempted to shift the EUREGIO’s work from purely project-based contacts towards a programmatic collaboration.

An important event was the establishment of the Alfred-Mozer-Kommission in 1970, responsible for cross-border initiatives in the cultural field. At the same time, a secretariat was established comprising two units on each side of the border and funded by membership fees. Two studies, in the fields of culture and economic affairs respectively, gave the secretariats a programmatic basis for the further development of the EUREGIO.

In the mid-seventies, the Arbeitsgruppe was given a formal basis by means of a statute (Satzung), and a common action programme was developed. This institutionalisation process ended with the Euregio-Rat in 1978, the first cross-border regional parliamentary assembly in Europe constituted by the political delegates of the member authorities.

In 1985, the separate secretariats were merged into a single Geschäftsstelle, located in Gronau (DE), situated 75m from the border, employing both Dutch and German staff. On the programmatic side, a ‘regional cross-border action programme’ was presented in 1987, outlining the general strategy for the EUREGIO for a twenty-year period (NEI nd). The elaboration of the programme was based on an agreement in 1984 between the German Federal ministry of economics, the Dutch ministry of economics and the German states of North Rhine-Westphalia and Lower Saxony. Funding was also provided by the European Commission. A steering committee was established, involving the partners in this agreement as well as the provinces, Bezirksregierungen and the EUREGIO. The action programme contained an economic and social analysis of the programme areas, and listed a series of measures aimed at promoting their socio-economic development.

This action programme constituted the main input for the first OP for the period 1989-1992, funded as pilot project under art. 10 ERDF. When the European Commission launched Interreg I in 1990, the EUREGIO reacted with the speedy elaboration of a second OP based on its accumulated experience."""

print(len(t.split()))

541


In [59]:
t = """
The Angle Peninsula Coast on the southern side of the entrance to the Milford Haven Waterway in Pembrokeshire, Wales, is a Site of Special Scientific Interest.[1] There is a wide range of wildlife and a former Royal Air Force station.


Remains of gun emplacements
The peninsula is rich in Second World War defences and the site of the former air base RAF Angle. The hollows in the banks around it were used to house machine guns in the Second World War and there was a searchlight battery here. Inland from East Picket bay are the remnants of the E-Pens used to house fighter aircraft if they were needed. In a field close to the World War I memorial there are the remains of an anti-aircraft post. On a section of the coastal path just past the RNLI lifeboat house there are visible remains of an anti-aircraft post. This site was later changed and used to house a 40 mm Rolls-Royce cannon. At the north hill, there are remains of a Laing hut that was used as housing for a searchlight. On a rocky patch of ground at west pill is a brick mine watcher hut. This was used specifically to watch out for the enemy who may be laying mines in Milford Haven.

RAF Angle was an airfield during World War II. It opened in 1941 after Luftwaffe attacks against Pembroke Dock. The airfield, which began as a station for No. 10 Group, Fighter Command, housed several squadrons during the war such as No. 312 (Czechoslovak) Squadron RAF and the Canadian 412 Transport Squadron. Planes included Supermarine Spitfires, Westland Whirlwinds and Hawker Hurricanes. In 1943 operational control passed on to the Fleet Air Arm of the Royal Navy. During this time a Sunderland flying boat landed at Angle airfield after receiving hull damage during a rescue. It returned to the RAF and became home to the Coastal command unit who tested weapons that could be used against German U-boats. After the war was over the buildings were no longer used and many were removed in the 1980s; however, some still stand in remote locations.

On 15 February 1996, the oil tanker Sea Empress grounded at the Milford Haven Waterway entrance, spilling 72,000 tonnes of crude oil. The coastline around Angle was severely damaged. The effect of the oil spill lasted several years and cost £60 million.[2]
"""
print(len(t.split()))

396


## wikiから特徴抽出
- 各カテゴリあたり、固有名詞50個について特徴を抽出する。
- まずカテゴリ毎に固有名詞をシャッフルする。
- そして、最初の固有名詞から順番に特徴を抽出していく。
- 特徴が60個取れなかった固有名詞は捨て、次の固有名詞に移る。
- これを繰り返し、特徴が60個取れた固有名詞を50個集める。
- 固有名詞が50個集まった時点で、そのカテゴリの特徴抽出は終了とする。

In [7]:

# 保存先
gen_result_savedir = os.path.join(project_root, "data", "generated_facts_in_wiki")
os.makedirs(gen_result_savedir, exist_ok=True)


print_results = True  # get_concept_list_with_complete_cos_similar_termsの結果を表示するかどうか


def fetch_wikipedia_page(title: str, lang: str = "en") -> dict:
    """Fetch a Wikipedia page by title and language.
    wiki pageを取得する．
    Args:
        title (str): Wikipediaページのタイトル
        lang (str): Wikipediaの言語コード（例: "en", "ja"）

    Returns:
        dict: ページ情報を含む辞書．ページが存在しない場合は'exists'キーがFalseとなる．

    Note:
        * titleに完全一致するページのみが取得されるため，google検索のように最も近いが異なるものを検索結果として返すことはない．
        * そのため，提示した固有名詞とは違う情報のページが返されることは無い．
        * また，titleの綴りが間違っていても, userがよく間違う綴りの場合は, wikipedia側で正しいページにリダイレクトしてくれる．
            * 例: "Colloseum" -> "Colosseum"のページを取得
    
    """
    wiki = wikipediaapi.Wikipedia(
        user_agent="my-wikipedia-fetcher/1.0 (contact: your_email@example.com)",
        language=lang,
    )
    page = wiki.page(title)

    if not page.exists():
        return {"exists": False, "title": title, "lang": lang}

    return {
        "exists": True,
        "title": page.title,
        "url": page.fullurl,
        "summary": page.summary,
        "text": page.text,  # 全文（長いので注意）
    }


def extract_wiki_main_text(text: str) -> str:
    """Wikipediaページの情報から本文テキストのみを抽出する。
    Args:
        text (str): Wikipediaページの全文

    Returns:
        str: Wikipediaページの本文テキスト。ページが存在しない場合は空文字列を返す。
    """

    STOP_SECTIONS = {
        "see also",
        "references",
        "notes",
        "further reading",
        "external links",
        "bibliography",
        "sources",
        "works cited",
        "citations",
    }
    # 見出し候補を | で連結
    titles = "|".join(re.escape(x) for x in STOP_SECTIONS)

    # 行頭に単独で出る見出しを検出
    # 例:
    # See also
    # References
    #
    # または wiki風:
    # == See also ==
    pattern = rf"(?mi)^\s*(?:=+\s*)?(?:{titles})(?:\s*=+)?\s*$"

    # 最初に見つかった見出し以降を全て削除
    m = re.search(pattern, text)
    if m:
        text = text[:m.start()]

    # 脚注番号 [1], [23] を削除
    text = re.sub(r"\[\d+\]", "", text)

    # 空行を整理
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text





In [ ]:
"""
作成日: 2025/12/25

wiki記事から, 固有名詞に特有の特徴を, 指定した形式に沿って抽出する
* ただし, 抽出した特徴文はそのまま保存し, 後で解析する.
* また, wikipageを取得したconceptのみに対して生成を実行. wikipageは，cossimが-0.2~0.9まで全て揃っているconceptのみを対象に収集しているため.

この生成にかかるコスト概算:
* 入力promptは520tokens, 出力は1300tokens程度と想定.
* 対象concept数は57->60件と想定
* json形式の指定は省略
* (0.25 / 1000000 *520 + 2 / 1000000 *1300) * 60 = 約0.16ドル. 激安!

Usage:
    nohup uv run python src/gen_factual_feature_sentences_from_wiki.py > gen_factual_feature_sentences_from_wiki.log 2>&1 &
"""

print("=== Generating factual feature sentences from Wikipedia articles ===")


# *** 準備 ***
# prompt と jsonschema を読み込み
print("Preparing prompts and schemas...")
print("Preparing prompts and schemas...")
prompt_schema_path = os.path.join(project_root, "data", "prompt", "baseprompt_of_gen_wiki_fact_extraction_for_googleai.json")
prompt_text_path = os.path.join(project_root, "data", "prompt", "baseprompt_of_gen_wiki_fact_extraction.txt")


with open(prompt_schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)
with open(prompt_text_path, "r", encoding="utf-8") as f:
    gen_triplet_prompt_base = f.read()




=== Generating factual feature sentences from Wikipedia articles ===
Preparing prompts and schemas...
Preparing prompts and schemas...
Checking already generated concepts...


### お試し生成

In [9]:
# wikipageを取得する
wiki_word_num_threshold = 500 # 十分な情報量のあるwikipageに対してのみ生成を行う

target_categories = filtered_category_count_df["class_label"].tolist()
print("=== Target categories for generating factual feature sentences ===")
print(target_categories)


=== Target categories for generating factual feature sentences ===
['Algorithm', 'Archive', 'Band', 'Food', 'Intercommunality', 'Legal Case', 'Medicine', 'Noble family', 'Painting', 'Port', 'Site of Special Scientific Interest', 'Wikimedia template', 'Windmill', 'academic subject', 'activity', 'agent', 'aircraft', 'airline', 'airport', 'album', 'anatomical structure', 'animal', 'archipelago', 'artery', 'artwork', 'asteroid', 'award', 'board game', 'book', 'bridge', 'broadcast network', 'broadcaster', 'brown dwarf', 'building', 'card game', 'casino', 'castle', 'cave', 'cemetery', 'church', 'city', 'company', 'contest', 'cycling race', 'dam', 'deity', 'drama', 'drug', 'educational institution', 'event', 'factory', 'family', 'fictional character', 'film festival', 'galaxy', 'game', 'garden', 'given name', 'glacier', 'golf course', 'government agency', 'handball team', 'historical period', 'hockey team', 'hot spring', 'hotel', 'identifier', 'ideology', 'island', 'lake', 'language', 'law fi

In [10]:

i = 8
print(target_categories[i])
concepts = whole_df[whole_df["class_label"] == target_categories[i]]["label"].tolist()
print(f"=== Example concepts in the category '{target_categories[i]}' ===")
print(concepts)

Painting
=== Example concepts in the category 'Painting' ===
['Madonna of Kyiv', 'Black Madonna of Częstochowa', 'My God, Help Me to Survive This Deadly Love', 'Rock Paintings of Hua Mountain', 'Along the River During the Qingming Festival', 'Our Lady of Kazan', 'The Battle of Anghiari', 'Our Lady of Perpetual Help', 'Trinity icon', 'Trojeručica', 'Balloon Girl', 'Christ Pantocrator (Sinai)', 'Leda and the Swan', 'Panagia Portaitissa', 'Madonna of the Yarnwinder', 'Wall of Love', 'Blue Dancers', 'Young Hare', 'Arrival of the Hungarians', 'Pornocrates', 'Portrait of a Young Man', 'San Damiano cross', 'Cestello Annunciation', 'The Chocolate Girl', 'Our Lady of Smolensk', 'Madonna and Child Enthroned with Angels and Prophets. ˋSanta Trinità Maestà´', 'Death playing chess', 'Beethoven Frieze', 'Great Piece of Turf', 'Agiosoritissa', 'Madonna Bardi', 'Theotokos of Tikhvin', 'Duria Antiquior', 'Saint Sebastian', 'Theotokos of Pochayiv', 'Manoppello Image', 'The Annunciation', 'Wing of a Euro

In [11]:
target_category = 'Painting'
target_concepts = whole_df[whole_df["class_label"] == target_category]["label"].tolist()
target_concept = target_concepts[21]
# target_concept = "Portsmouth Harbour"
print(target_concept)

wiki_info = fetch_wikipedia_page(target_concept, lang="en")
main_text = extract_wiki_main_text(wiki_info['text'])
print(main_text)
wiki_info

San Damiano cross
The San Damiano Cross is the large Romanesque rood cross before which St. Francis of Assisi was praying when he is said to have received the commission from the Lord to rebuild the Church.  It hangs in the Basilica of Saint Clare (Basilica di Santa Chiara) in Assisi, Italy, with a replica in its original position in the church of San Damiano nearby.  Franciscans cherish this cross as the symbol of their mission from God.
The cross is a crucifix of a type sometimes called an icon cross because in addition to the main figure of the Christ, it contains images of other saints and people related to the incident of Christ's crucifixion.  The tradition of such painted crucifixes began in the Eastern Church and possibly reached Italy via Montenegro and Croatia.

History
The San Damiano Cross was one of a number of crosses painted with similar figures during the 11th century in Umbria. The name of the painter is unknown, but it was made around the year 1100. The purpose of an 

{'exists': True,
 'title': 'San Damiano Cross',
 'url': 'https://en.wikipedia.org/wiki/San_Damiano_Cross',
 'summary': "The San Damiano Cross is the large Romanesque rood cross before which St. Francis of Assisi was praying when he is said to have received the commission from the Lord to rebuild the Church.  It hangs in the Basilica of Saint Clare (Basilica di Santa Chiara) in Assisi, Italy, with a replica in its original position in the church of San Damiano nearby.  Franciscans cherish this cross as the symbol of their mission from God.\nThe cross is a crucifix of a type sometimes called an icon cross because in addition to the main figure of the Christ, it contains images of other saints and people related to the incident of Christ's crucifixion.  The tradition of such painted crucifixes began in the Eastern Church and possibly reached Italy via Montenegro and Croatia.",
 'text': 'The San Damiano Cross is the large Romanesque rood cross before which St. Francis of Assisi was praying

In [12]:
# *** prompt構築 ***
prompt = gen_triplet_prompt_base.replace("<target_concept>", target_concept)
prompt = prompt.replace("<wikipedia_text>", main_text[:10000])  # wikiページ本文は10000文字までに制限 (例. "Portsmouth Harbour"の本文は文字数9498, 単語数は1538)

print(prompt)


You are given the full main text of a Wikipedia article describing a specific proper noun.

Your task is to extract factual characteristics of the target object using only the information explicitly stated or clearly implied in the provided Wikipedia text.

### Input

#### Target object
San Damiano cross

#### Wikipedia article text
The San Damiano Cross is the large Romanesque rood cross before which St. Francis of Assisi was praying when he is said to have received the commission from the Lord to rebuild the Church.  It hangs in the Basilica of Saint Clare (Basilica di Santa Chiara) in Assisi, Italy, with a replica in its original position in the church of San Damiano nearby.  Franciscans cherish this cross as the symbol of their mission from God.
The cross is a crucifix of a type sometimes called an icon cross because in addition to the main figure of the Christ, it contains images of other saints and people related to the incident of Christ's crucifixion.  The tradition of such pa

### 生成

In [51]:
print(prompt)


You are given the full main text of a Wikipedia article describing a specific proper noun.

Your task is to extract factual characteristics of the target object using only the information explicitly stated or clearly implied in the provided Wikipedia text.

### Input

#### Target object
mural of Marcus Rashford

#### Wikipedia article text
In 2020, a mural of footballer Marcus Rashford by street artist Akse P19 was painted in the Withington area of Manchester, United Kingdom. The mural was created in recognition of the work Rashford did during the COVID-19 pandemic in the United Kingdom to help tackle child food poverty.
After Rashford had missed a penalty kick for England in the UEFA Euro 2020 Final in July 2021, the mural was vandalised, prompting locals to post messages of support for Rashford before its restoration.

Description
Based on a photograph by Daniel Cheetham, the painting of Marcus Rashford was completed in 2020 by street artist Akse, in collaboration with the street ar

In [54]:
model = "gemini-2.5-flash-lite"
response = client.models.generate_content(
    model=model,
    contents=prompt,
    config=schema
)



In [55]:
import json

record = {
    "response_id": response.response_id,
    "model_version": response.model_version,
    "text": response.text,
    "parsed": response.parsed,
    "usage_metadata": {
        "prompt_token_count": response.usage_metadata.prompt_token_count,
        "candidates_token_count": response.usage_metadata.candidates_token_count,
        "total_token_count": response.usage_metadata.total_token_count,
    },
}


filename = f"{target_concept.replace(' ', '_')}.json"
with open(os.path.join(gen_result_savedir, filename), "w", encoding="utf-8") as f:
    # f.write(json.dumps(record, ensure_ascii=False) + "\n") # jsonl形式で1fileに追加し続ける場合 ('w'ではなく'a'にする必要あり)
    json.dump(record, f, ensure_ascii=False, indent=4)

In [ ]:
# お試しで1カテゴリ分を生成する
# num = 10
target_category = 'Painting'
target_concepts = whole_df[whole_df["class_label"] == target_category]["label"].tolist()
for target_concept in target_concepts:
    print(target_concept)
    if target_concept in generated_concepts:
        print(f"Concept '{target_concept}' ALREADY has GENERATED factual feature sentences. Skipping generation.")
        continue
    # wikipageのテキストをapiで取得する
    wiki_info = fetch_wikipedia_page(target_concept, lang="en")
    if wiki_info["exists"] == False:
        print(f"Wikipedia page for concept '{target_concept}' DOES NOT exist. Skipping generation.")
        continue
    main_text = extract_wiki_main_text(wiki_info['text'])
    print(main_text)
    
    if len(main_text.split()) < wiki_word_num_threshold:
        # もし十分な情報量(本文の文字数)がない場合は、生成をスキップする
        print(f"Concept '{target_concept}' has insufficient Wikipedia text ({len(main_text.split())} words). Skipping generation.")
        continue

    

    

### 生成2: 1カテゴリ分だけ生成

In [ ]:
def gen_with_google_genai_api(prompt, schema, temperature=0.2, topP=0.8, model="gemini-2.5-flash-lite", max_retries=1):
    """Google GenAI APIを呼び出して、指定したスキーマに沿った応答を生成する。
    Args:
        prompt (str): プロンプトのテキスト。
        schema (dict): 生成された応答が従うべきJSONスキーマ。
        temperature (float): 生成時の温度パラメータ。デフォルトは0.2。
        topP (float): 生成時のtopPパラメータ。デフォルトは0.8。
        model (str): 使用するモデル名。デフォルトは"gemini-2.5-flash-lite"。
        max_retries (int): API呼び出しの最大リトライ回数。デフォルトは1。
    Returns:
        response をそのまま返す
        ×dict: 生成された応答がスキーマに従っている場合はその応答を辞書形式で返す。スキーマに従っていない場合はNoneを返す。
    """
    schema["temperature"] = temperature  # スキーマに温度パラメータを追加
    schema["topP"] = topP  # スキーマにtopPパラメータを追加

    # *** Gemini で生成 ***
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=schema
            )
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            if attempt == max_retries - 1:
                raise e
            else:
                continue
        # APIからの応答を取得
        return response
        # generated_text = response.output_text.strip()
        # # print(f"【generated_text】\n{generated_text}")
        # return generated_text
    return None


def gen_with_openai_api(prompt, schema, temperature, topP, model="gpt-5-mini", max_retries=1):
    """OpenAI APIを呼び出して、指定したスキーマに沿った応答を生成する。
    Args:
        prompt (str): プロンプトのテキスト。
        schema (dict): 生成された応答が従うべきJSONスキーマ。
        temperature (float): 生成時の温度パラメータ。デフォルトは0.2。
        topP (float): 生成時のtopPパラメータ。デフォルトは0.8。
        model (str): 使用するモデル名。デフォルトは"gpt-5-mini"。
        max_retries (int): API呼び出しの最大リトライ回数。デフォルトは1。
    Returns:
        response をそのまま返す
        ×dict: 生成された応答がスキーマに従っている場合はその応答を辞書形式で返す。スキーマに従っていない場合はNoneを返す。
    """

    # *** GPT5で生成 ***
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=model,
                input=prompt,
                text={
                    "format": schema
                },
            )
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            if attempt == max_retries - 1:
                raise e
            else:
                continue
        # APIからの応答を取得
        return response
        # generated_text = response.output_text.strip()
        # # print(f"【generated_text】\n{generated_text}")
        # return generated_text
    return None



In [59]:


def gen_and_save_wiki_fact_extraction_for_concept(
        target_concept, 
        feat_num_threshold, 
        wiki_word_num_threshold, 
        prompt_template, 
        schema, 
        gen_result_savedir, 
        model="gemini-2.5-flash-lite", 
        temperature=0.2, 
        topP=0.8,
        max_retries=1
    ):
    """
    Generate factual feature sentences for a given concept using Wikipedia information.
    Args:
        target_concept (str): The concept for which to generate factual features.
        feat_num_threshold (int): The minimum number of valid features required for successful generation.
        wiki_word_num_threshold (int): The minimum number of words required in the Wikipedia text for successful generation.
        prompt_template (str): The template for the generation prompt, containing placeholders for the target concept and Wikipedia text.
        schema (dict): The JSON schema that the generated response should adhere to.
        gen_result_savedir (str): The directory where the generated results should be saved.
        model (str): The model to use for generation. Default is "gemini-2.5-flash-lite".
        temperature (float): The temperature parameter for generation. Default is 0.2.
        topP (float): The topP parameter for generation. Default is 0.8.
        max_retries (int): The maximum number of retries for API calls. Default is 1.
    Returns:
        bool: True if generation is successful (i.e., the number of valid features meets the threshold), False otherwise.

    もし生成された特徴の数(unkonwn除く)がfeat_num_threshold未満であっても、生成結果自体は保存する。
    """
    
    # wikipageのテキストをapiで取得する
    wiki_info = fetch_wikipedia_page(target_concept, lang="en")
    if wiki_info["exists"] == False:
        print(f"Wikipedia page for concept '{target_concept}' DOES NOT exist. Skipping generation.")
        return False
    main_text = extract_wiki_main_text(wiki_info['text'])
    # print(main_text)
    
    if len(main_text.split()) < wiki_word_num_threshold:
        # もし十分な情報量(本文の文字数)がない場合は、生成をスキップする
        print(f"Concept '{target_concept}' has insufficient Wikipedia text ({len(main_text.split())} words). Skipping generation.")
        return False

    # prompt構築
    prompt = prompt_template.replace("<target_concept>", target_concept)
    prompt = prompt.replace("<wikipedia_text>", main_text[:10000])  # wikiページ本文は10000文字までに制限 (例. "Portsmouth Harbour"の本文は文字数9498, 単語数は1538)

    # 生成
    if model.startswith("gemini"):
        response = gen_with_google_genai_api(prompt, schema, temperature, topP, model, max_retries)
        if response is not None:
            record = {
                "response_id": response.response_id,
                "model_version": response.model_version,
                "text": response.text,
                "parsed": response.parsed,
                "usage_metadata": {
                    "prompt_token_count": response.usage_metadata.prompt_token_count,
                    "candidates_token_count": response.usage_metadata.candidates_token_count,
                    "total_token_count": response.usage_metadata.total_token_count,
                },
            }
            # 生成された(有効な)特徴数が feat_num_threshold 以上であれば生成成功とみなす
            total_features = sum(response.parsed['english'].values(), [])
            total_features = [feat for feat in total_features if feat.lower() != "unknown"] # featが unknown や Unknown などの意味のない特徴であれば削除する
            if len(total_features) >= feat_num_threshold:
                gen_succeeded = True
            else:
                gen_succeeded = False
        else:
            record = {
                "error": "Failed to generate a valid response after maximum retries.",
                "prompt": prompt,
                "schema": schema,
                "model": model,
                "max_retries": max_retries,
            }
            gen_succeeded = False

    elif model.startswith("gpt"):
        response = gen_with_openai_api(prompt, schema, temperature, topP, model, max_retries)
        pass # [WIP] ここではまだgpt-5-miniでの生成は試さないため、実装は保留中
    else:
        raise ValueError(f"Unsupported model: {model}")
    
    # 保存
    filename = f"{target_concept.replace(' ', '_')}.json"
    with open(os.path.join(gen_result_savedir, filename), "w", encoding="utf-8") as f:
        # f.write(json.dumps(record, ensure_ascii=False) + "\n") # jsonl形式で1fileに追加し続ける場合 ('w'ではなく'a'にする必要あり)
        json.dump(record, f, ensure_ascii=False, indent=4)

    return gen_succeeded

In [61]:
# 生成済みのconceptについては生成をスキップするため、事前に確認しておく
print("Checking already generated concepts...")
generated_concepts = []
for filename in os.listdir(gen_result_savedir):
    if filename.endswith(".json"):
        concept = filename.split('.json')[0]  # 拡張子を除いた部分を取得
        concept = concept.replace('_', ' ')  # ファイル名のアンダースコアをスペースに変換
        generated_concepts.append(concept)

generated_concepts

Checking already generated concepts...


['Silent Majority',
 'The History of Mexico',
 'Ustyug Annunciation',
 'mural of Marcus Rashford',
 "Noah's Ark",
 'Dormition of Theotokos',
 'Sutton twin towns mural',
 'Leda and the Swan',
 'Adoration of the Kings']

In [ ]:
import random
random.seed(42)  # 再現性のためにシードを固定

target_category = 'Painting'
target_concepts = whole_df[whole_df["class_label"] == target_category]["label"].tolist()
target_concepts = random.sample(target_concepts, k=len(target_concepts))


model = "gemini-2.5-flash-lite"
max_retries = 1
feat_num_threshold = 60 # 1概念あたりで、生成された特徴の数がこの数以上であれば生成成功とみなす
wiki_word_num_threshold = 500 # 十分な情報量のあるwikipageに対してのみ生成を行うための、wikipageの本文の単語数の閾値
propnoun_num_for_new_concept = 50
propnoun_num_for_init_vec = 100
temperature = 0.2
topP = 0.8


# 生成済みのconceptについては生成をスキップするため、事前に確認しておく
print("Checking already generated concepts...")
generated_concepts = []
for filename in os.listdir(gen_result_savedir):
    if filename.endswith(".json"):
        concept = filename.split('.json')[0]  # 拡張子を除いた部分を取得
        concept = concept.replace('_', ' ')  # ファイル名のアンダースコアをスペースに変換
        generated_concepts.append(concept)


# 生成ループ
gen_success_count = 0   # 生成成功した概念の数 (生成された特徴の数が feat_num_threshold 以上であった概念の数)
total_gen_count = 0     # 生成を試みた概念の総数 (生成成功・失敗に関わらず)
for target_concept in target_concepts:
    if target_concept in generated_concepts:
        print(f"Concept '{target_concept}' ALREADY has GENERATED factual feature sentences. Skipping generation.")
        continue
    if gen_success_count >= propnoun_num_for_new_concept:
        print(f"REACHED the target number of successfully generated concepts ({propnoun_num_for_new_concept}). Stopping generation for this category.")
        break


    total_gen_count += 1
    print(f"Generating factual feature sentences for concept: {target_concept}")
    gen_succeeded = gen_and_save_wiki_fact_extraction_for_concept(
        target_concept=target_concept,
        feat_num_threshold=feat_num_threshold,
        wiki_word_num_threshold=wiki_word_num_threshold,
        prompt_template=gen_triplet_prompt_base,
        schema=schema,
        gen_result_savedir=gen_result_savedir,
        model=model,
        temperature=temperature,
        topP=topP,
        max_retries=max_retries
    )
    print(f"\tGeneration result for {target_concept}: {gen_succeeded}")
    gen_success_count += gen_succeeded

print(f"{gen_success_count}/{total_gen_count} concepts successfully generated with factual feature sentences for the category {target_category}.")

Checking already generated concepts...
Concept 'mural of Marcus Rashford' ALREADY has GENERATED factual feature sentences. Skipping generation.
Generating factual feature sentences for concept: Boy Cutting Grass with a Sickle
Concept 'Boy Cutting Grass with a Sickle' has insufficient Wikipedia text (24 words). Skipping generation.
	Generation result for Boy Cutting Grass with a Sickle: False
Concept 'Leda and the Swan' ALREADY has GENERATED factual feature sentences. Skipping generation.
Generating factual feature sentences for concept: Independence and the Opening of the West
Concept 'Independence and the Opening of the West' has insufficient Wikipedia text (134 words). Skipping generation.
	Generation result for Independence and the Opening of the West: False
Generating factual feature sentences for concept: Art Buff
Concept 'Art Buff' has insufficient Wikipedia text (206 words). Skipping generation.
	Generation result for Art Buff: False
Generating factual feature sentences for conc

In [66]:
target_category

'Painting'

In [63]:
gen_success_count

50

In [65]:
len(os.listdir(gen_result_savedir))

70